# journey

> A goal and a path to it: an ordered list of steps, one concept each, that the learner can see and change.

A learner often wants to learn more than one concept. "I want to learn how LLMs work" needs tokenizers, embeddings, attention,
transformers and more, in a given order. A journey keeps that goal and that order on the learning shelf.

The planner (one model call) proposes the steps. It uses the concepts on the shelf when they fit. It makes new concepts for
the topics that the shelf does not have. Then the learner can change the path: move, skip, mark as known, add and split steps.

The journey does not keep progress. Progress comes from the learner model each time. A step is learned when the floor of its concept
reaches the goal depth. So a concept that the learner learns anywhere counts in each journey that has it.

In [ ]:
#| default_exp journey

In [ ]:
#| hide
import os, tempfile
from fastcore.test import *
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

Import the modules that this notebook uses.

In [ ]:
#| export
import json, math, re, random, threading, time, uuid
from fastcore.utils import *
from abhyasa.store import Learning, slugify, _src, clean_subject
from abhyasa.model import learned
from abhyasa.settings import val
from abhyasa.llm import ask_json, for_task
from abhyasa.prompts import register, prompt
from abhyasa.sources import extract, _index, doc_pages, from_vault, placement, is_id_title, ARXIV_LINK, ARXIV_ID, arxiv_title
import abhyasa.profile  # adds Learning.profile
from abhyasa.method import check_for, EXAMPLES_TO, WALK_WAYS, TEXT_WAYS, CHECK_KIND
from abhyasa.ground import ground, ground_text

## The document

A journey is one vault document of kind `journey`. Its meta has the goal, a title, a subject, and the steps. Each step has a slug,
a status (`todo`, `known` or `skip`) and the reason that it is in the path.

In [ ]:
#| export
STATUS = ('todo', 'known', 'skip')

def _jmeta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def put_journey(self:Learning, j:dict) -> AttrDict:
    "Save journey `j`, and replace an earlier one with the same id."
    j = {k: v for k, v in dict(j).items() if k != 'doc_id'}
    j.update(updated=time.time(), steps=[dict(s) for s in j.get('steps', [])])
    names = [(self.concept(s['slug']) or {}).get('name', s['slug']) for s in j['steps']]
    text = f"Journey: {j['title']} ({j['id']})\n\nGoal: {j.get('goal', '')}\n\nSteps: {', '.join(names)}"
    r = self.v.add(text, title=f"Journey: {j['title']}", source=_src('journey', j['id']), kind='journey', meta=j)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **j)
    return self.journey(j['id'])

@patch
def journey(self:Learning, jid:str) -> AttrDict|None:
    "One journey by id."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('journey', jid)]))
    return _jmeta(r) if r else None

@patch
def journeys(self:Learning) -> L:
    "All journeys, the most recently changed first."
    return L(sorted(map(_jmeta, self._docs('journey')), key=lambda j: -j.get('updated', 0)))

@patch
def drop_journey(self:Learning, jid:str):
    "Remove a journey. Its concepts and their state stay."
    if (j := self.journey(jid)): self.v.forget(j.doc_id)

## Progress

`step_state` names the state of one step from its status and its concept. The states `learned`, `known`, `skipped` and
`missing` (the concept is gone) are done. The states `new` and `started` are not done.

In [ ]:
#| export
DONE = ('learned', 'known', 'skipped', 'missing')

def step_state(c, status:str='todo', goal:int=None) -> str:
    "One of new, started, learned, known, skipped or missing."
    if status == 'known': return 'known'
    if status == 'skip': return 'skipped'
    if c is None: return 'missing'
    if learned(c, goal): return 'learned'
    return 'started' if c.get('seen') else 'new'

def progress(lr:Learning, j, goal:int=None, by:dict=None) -> AttrDict:
    "Each step with its concept and state, the count done, the count that counts, and the current step."
    by, goal = by if by is not None else {c.slug: c for c in lr.concepts()}, goal or val('checks.goal')
    steps = [AttrDict(s, c=by.get(s['slug']), name=(by.get(s['slug']) or {}).get('name', s['slug']),
                      state=step_state(by.get(s['slug']), s.get('status', 'todo'), goal)) for s in j['steps']]
    cur = first(s for s in steps if s.state not in DONE)
    return AttrDict(steps=steps, done=sum(s.state in ('learned', 'known') for s in steps),
                    total=sum(s.state not in ('skipped', 'missing') for s in steps), current=cur, finished=cur is None and bool(steps))

`current` gives the slug of the step to learn now. A step in `skip` waits, unless no other step is left.

In [ ]:
#| export
def current(lr:Learning, j, skip=(), goal:int=None) -> str|None:
    "The slug of the first step that is not done and not in `skip`; else the first step that is not done."
    left = [s.slug for s in progress(lr, j, goal).steps if s.state not in DONE]
    return first(s for s in left if s not in skip) or first(left)

## Change the path

`journey_step` makes one change to one step. `journey_add` adds a step at the end, or before another step.

In [ ]:
#| export
@patch
def journey_step(self:Learning, jid:str, slug:str, act:str) -> AttrDict:
    "Do `act` on step `slug`: up, down, skip, known, todo or remove."
    j = self.journey(jid)
    if j is None: raise KeyError(f'no journey {jid!r}')
    ss = j.steps
    i = next((k for k, s in enumerate(ss) if s['slug'] == slug), None)
    if i is None: return j
    if act == 'up' and i > 0: ss[i - 1], ss[i] = ss[i], ss[i - 1]
    elif act == 'down' and i < len(ss) - 1: ss[i + 1], ss[i] = ss[i], ss[i + 1]
    elif act in STATUS: ss[i]['status'] = act
    elif act == 'remove': ss.pop(i)
    return self.put_journey(j)

@patch
def journey_add(self:Learning, jid:str, slug:str, before:str=None, why:str='') -> AttrDict:
    "Add concept `slug` as a step, at the end or before step `before`. A step that is there already stays where it is."
    j = self.journey(jid)
    if j is None: raise KeyError(f'no journey {jid!r}')
    if any(s['slug'] == slug for s in j.steps): return j
    at = next((k for k, s in enumerate(j.steps) if s['slug'] == before), len(j.steps))
    j.steps.insert(at, dict(slug=slug, status='todo', why=why))
    return self.put_journey(j)

## The planner

The planner gets the goal and the learner profile as text. It also gets the concepts on the shelf that are near the goal or that the
learner started. It returns the steps in order.

In [ ]:
#| export
PLAN_SP = register('walk.plan', 'Path planner', 'How the tutor plans the steps toward a goal, for a journey or a walk on a topic.', """You plan a learning path toward one learner's goal.
- Give 4 to 9 steps, in the order to learn them. Each step is one idea that a learner can be asked about and can explain.
- Start where the learner is. Leave out what they know well: their background, and their concepts at depth 4 or more.
- When a known concept fits a step, give its slug in `slug`. Else `slug` is "".
- `statement` is true and self-contained: one to three sentences. Maths in $...$.
- `why` says in one short sentence why the step is in the path. Write to the learner as "you", in ASD-STE100 Simplified Technical English.
- `needs` names the earlier steps that this step needs first.
- `ways` names up to 3 ways that suit the idea best, from: code, maths, analogy, example, diagram, words, demo, order.""")

def plan_prompt(goal:str, who:str='', known=()) -> str:
    "The prompt that plans a path to `goal`."
    kn = '\n'.join(f"- {s}: {n} (depth {f})" for s, n, f in known)
    return (f"Goal: {goal}\n" + (f"About the learner:\n{who}\n" if who else '')
            + (f"Concepts the learner has (slug: name (depth known)):\n{kn}\n" if kn else '')
            + '\nReturn JSON: {"title": str (3 to 6 words), "subject": str (one lower-case word, such as ml), '
            '"steps": [{"name": str, "statement": str, "slug": str, "why": str, "needs": [str], "ways": [str]}]}')

def near_goal(lr:Learning, goal:str, k:int=40, most:int=150) -> list:
    "`(slug, name, floor)` for the concepts that search finds for `goal`, and the ones the learner started."
    ids = {h['doc_id'] for h in lr.search(goal, kind='concept', limit=k)}
    return [(c.slug, c.name, c.floor) for c in lr.concepts() if c.doc_id in ids or c.get('seen')][:most]

`plan` saves the journey. A step can name a known slug, or a known name or alias. Then it uses that concept and keeps its state.
Each other step becomes a new concept, with the steps it needs as its prereqs. With `sources`, `plan` links each new concept to the
passages that name it in the learner's own sources.

In [ ]:
#| export
def _new_id(title): return f"{slugify(title)[:40]}-{uuid.uuid4().hex[:4]}"

def plan(lr:Learning, ask, goal:str, who:str='', sources=None) -> AttrDict:
    "Plan a journey to `goal`, make the concepts that the shelf does not have, and save it."
    r = ask_json(for_task(ask, 'plan'), plan_prompt(goal, who, near_goal(lr, goal)), prompt('walk.plan'))
    idx, subject, steps, slug_of = _index(lr), (r.get('subject') or '').lower().strip(), [], {}
    for s in r.get('steps') or []:
        slug = s.get('slug') if s.get('slug') and lr.concept(s['slug']) else idx.get(s['name'].lower())
        if not slug:
            slug = slugify(s['name'])
            needs = [slug_of[n.lower()] for n in s.get('needs') or () if n.lower() in slug_of]
            lr.put(s['name'], s.get('statement', ''), subject, slug=slug, prereqs=needs, ways=[w for w in s.get('ways') or () if w in WALK_WAYS], provenance=[dict(src='journey', goal=goal)])
            idx[s['name'].lower()] = slug
            if sources:
                try: lr.connect(slug, sources)
                except Exception: pass
        slug_of[s['name'].lower()] = slug
        if slug not in [x['slug'] for x in steps]: steps.append(dict(slug=slug, status='todo', why=s.get('why', '')))
    title = r.get('title') or goal[:60]
    return lr.put_journey(dict(id=_new_id(title), title=title, goal=goal, subject=subject, steps=steps, created=time.time()))

## Walks

A walk is a journey with `mode='walk'`. It is the reading path: one step for each idea, in reading order. Each step has the passage to
read (`read`) and the check to do (`check`). `progress` does not change.

`_check` picks the check of one concept. It uses the stats and the profile of the learner, as `Tutor.ask_on` does.
A tool walk (`tool=True`) never gets a code check: code and tools are learned in Leela.

In [ ]:
#| export
TOOL_WAYS = tuple(w for w in WALK_WAYS if w != 'code')

def _check(lr:Learning, c, tool:bool=False, hands:bool=True) -> AttrDict:
    "The check for concept `c`, from `check_for` with the stats and the profile of the learner. A tool walk has no code check. `hands=False` gives no hands-on check."
    p = lr.profile()
    if tool: c = AttrDict(c, ways=[w for w in c.get('ways') or TEXT_WAYS if w in TOOL_WAYS] or ['words'])
    return check_for(c, lr.stats(), random.Random(), p.get('explore', 'sometimes'), [EXAMPLES_TO[e] for e in p.examples if e in EXAMPLES_TO], hands=hands)

`short_check` gives a short-answer check in a text way. A walk uses it when a demo or an order check has no card. The way comes from the
text ways of the concept, or from all text ways. A code way is not in the list, because its check is a code task.

In [ ]:
#| export
SHORT_WAYS = tuple(w for w in TEXT_WAYS if CHECK_KIND.get(w, 'short') == 'short')

def short_check(lr:Learning, c) -> AttrDict:
    "A short-answer check for concept `c`, in one of its text ways, or in any text way."
    return _check(lr, {**c, 'ways': [w for w in c.get('ways') or () if w in SHORT_WAYS] or list(SHORT_WAYS)})

A concept with a demo way and a code way gets a short check in another text way.

In [ ]:
from abhyasa.store import temp_learning
ckl = temp_learning()
ck = short_check(ckl, AttrDict(slug='x', name='X', ways=['demo', 'code']))
assert ck.kind == 'short' and ck.way in SHORT_WAYS and ck.depth == 2

### A walk has a most number of steps

A long source, such as a docs site, can have 60 ideas or more. A walk of that length is too long to finish, and each step costs model calls for its note and its check.
So a walk has at most `val('walks.steps')` steps (12 by default; you can change it on the Settings page). When a source has more ideas, `main_steps` asks the model one time to pick the main ideas, with the prompt `walk.pick`. You can change it on the Settings page.
When the reply has fewer valid numbers, the first ideas that are not picked fill the walk in source order. The steps keep the order of the source, and a prerequisite that is picked comes before the idea that needs it.
The pick happens when the walk is made, before any step writes a note or a check. The other ideas stay on the shelf as concepts.
The walk keeps their slugs in `more`, so the plan can say how many more ideas the source has.

In [ ]:
#| export
PICK_SP = register('walk.pick', 'Main ideas of a walk', 'How the tutor picks the main ideas of a long source for a walk. "Ideas in a walk" in Walks sets how many.', """You pick the main ideas of a source for a learner.
- Pick the ideas that the rest of the source builds on, and its main results.
- Leave out small details, and ideas that only name a part or an option.
- Give each idea by its number in the list.""")

def pick_prompt(cs, most:int) -> str:
    "The prompt that asks for the `most` main ideas of the concepts `cs`, numbered from 1 in the order of the source."
    ls = '\n'.join(f"{k}. {c.get('name')}: {c.get('statement', '')}" for k, c in enumerate(cs, 1))
    return f'Ideas, in the order of the source:\n{ls}\n\nPick at most {most} ideas. Return JSON: {{"pick": [int]}}'

def main_steps(lr:Learning, ask, steps:list, most:int=None) -> tuple:
    "The `most` main steps of `steps` (default: the setting `walks.steps`) that the model picks, filled up in source order when it picks fewer, and the slugs of the other steps. The steps keep their order, with each prerequisite first."
    most = most or val('walks.steps')
    if len(steps) <= most: return steps, []
    cs = [lr.concept(s['slug']) or AttrDict(name=s['slug']) for s in steps]
    try: r = ask_json(for_task(ask, 'pick'), pick_prompt(cs, most), prompt('walk.pick'))
    except ValueError: r = {}
    ks = [k - 1 for k in (r.get('pick') if isinstance(r, dict) else None) or () if type(k) is int and 1 <= k <= len(steps)]
    ks = list(dict.fromkeys(ks))[:most]
    keep = sorted(ks + [k for k in range(len(steps)) if k not in ks][:most - len(ks)])
    at, seen, out = {steps[k]['slug']: k for k in keep}, set(), []
    def put(k):
        if k in seen: return
        seen.add(k)
        for p in cs[k].get('prereqs') or ():
            if p in at: put(at[p])
        out.append(k)
    for k in keep: put(k)
    return [steps[k] for k in out], [s['slug'] for k, s in enumerate(steps) if k not in seen]

A demo, an order check or a code task takes longer than a short answer. So at most a share of the steps of a walk get one (the setting `walks.hands`, a third by default, rounded down, and at least 1 in a walk of 3 steps or more).
`cap_hands` counts the hands-on checks in walk order, after `main_steps`. When the walk has its share, each later step gets a check in its text ways. The count does not depend on chance, so a test can check it.

In [ ]:
#| export
def hands_most(n:int) -> int:
    "The most hands-on checks in a walk of `n` steps: the share `walks.hands` of them, rounded down, and at least 1 from 3 steps when the share is not 0."
    share = val('walks.hands')
    k = math.floor(n * share + 1e-9)
    return max(k, 1) if n >= 3 and share else k

def cap_hands(lr:Learning, steps:list, tool:bool=False) -> list:
    "`steps` with at most `hands_most` hands-on checks, the first ones in walk order. Each later step gets a check in its text ways."
    most, n = hands_most(len(steps)), 0
    for s in steps:
        if (s.get('check') or {}).get('kind', 'short') == 'short': continue
        n += 1
        if n > most: s['check'] = _check(lr, lr.concept(s['slug']) or AttrDict(slug=s['slug']), tool, hands=False)
    return steps

`journey_from_source` extracts the ideas of one document, and makes one step for each idea of that document, in the order of the document.
A step has the passage that `extract` kept. A concept that you know at depth 4 or more has the status `known`.

In [ ]:
#| export
def _leela(c, tool:bool) -> dict:
    "`leela=True` for an idea with a code way on a tool walk: its code is learned in Leela."
    return dict(leela=True) if tool and 'code' in (c.get('ways') or ()) else {}

def _subject_of(lr:Learning, slugs, subject:str='') -> str:
    "`subject`, or the most common subject of the concepts `slugs` when it is empty."
    ss = [c.subject for c in map(lr.concept, slugs) if c and c.subject]
    return subject or (max(set(ss), key=ss.count) if ss else '')

def _steps(lr:Learning, slugs, doc_id:str, shelf:str, title:str, tool:bool=False) -> list:
    "One walk step for each of `slugs` that has a place in document `doc_id`, in the order of the document. On a tool walk, an idea with a code way has `leela`."
    cs = sorted(((c, placement(c, doc_id)) for c in map(lr.concept, slugs) if c and placement(c, doc_id)), key=lambda x: x[1].get('order') or 0)
    return [dict(slug=c.slug, status='known' if learned(c) else 'todo', why='',
                 read=dict(shelf=shelf, doc_id=doc_id, title=title, section=p.get('section') or '',
                           page=p.get('page'), span=p.get('span'), text=p.get('passage') or '', figure=p.get('figure')),
                 check=_check(lr, c, tool), **_leela(c, tool)) for c, p in cs]

def _save_walk(lr:Learning, ask, jid:str, title:str, steps:list, source:dict|None, goal:str, subject:str, tool:bool=False, **extra) -> AttrDict:
    "Save a walk that is not started yet, with the main steps only (`main_steps`) and its share of hands-on checks (`cap_hands`). `tool` marks a library or a command-line tool."
    steps, more = main_steps(lr, ask, steps)
    steps = cap_hands(lr, steps, tool)
    return lr.put_journey(dict(id=jid, title=title, goal=goal, subject=subject, steps=steps, created=time.time(),
                               mode='walk', source=source, started=None, sitting=None, tool=tool, more=more, **extra))

def _walk(lr:Learning, ask, jid:str, title:str, slugs, doc_id:str, source:dict|None, goal:str, subject:str, tool:bool=False) -> AttrDict:
    "Save a walk with one step for each of `slugs` that has a place in document `doc_id`, in the order of the document."
    return _save_walk(lr, ask, jid, title, _steps(lr, slugs, doc_id, (source or {}).get('shelf', 'store'), title, tool), source, goal, subject, tool)

def journey_from_source(lr:Learning, ask, ref:str, shelf:str='store', subject:str='', tool:bool=False) -> AttrDict:
    "Make a walk from one vault document, by doc id, source or title. `tool` marks a library or a command-line tool."
    d, subject = doc_pages(lr, ref, shelf), clean_subject(subject)
    slugs = from_vault(lr, ask, d.doc_id, subject, shelf)
    return _walk(lr, ask, _new_id(d.title), d.title, slugs, d.doc_id, dict(shelf=shelf, doc_id=d.doc_id, title=d.title), f'Read {d.title}', _subject_of(lr, slugs, subject), tool)

`journey_from_text` does the same for pasted text. A passage shows only when its quote is in the text word for word.

In [ ]:
#| export
def journey_from_text(lr:Learning, ask, text:str, title:str, subject:str='', tool:bool=False) -> AttrDict:
    "Make a walk from pasted `text`."
    jid, subject = _new_id(title), clean_subject(subject)
    slugs = extract(lr, ask, subject=subject, source=dict(src='text', doc_id=jid, title=title), pages=[(None, text)])
    return _walk(lr, ask, jid, title, slugs, jid, None, f'Read {title}', _subject_of(lr, slugs, subject), tool)

`journey_from_sources` makes one walk across many pages, for example the pages of a docs site that `file_link` filed. The tutor reads each page in turn.
The steps come in page order, and in each page in the order of the page. An idea that two pages name gets one step, in the first page where its passage is.
When no page has its passage, the step is in the first page that names it. `progress(k, n)` hears of page `k` of `n` before the tutor reads it.

In [ ]:
#| export
def journey_from_sources(lr:Learning, ask, docs:list, title:str='', subject:str='', progress=None, tool:bool=False, **extra) -> AttrDict:
    "Make one walk across the vault documents `docs`, each with `shelf`, `doc_id`, `title` and `url`, in their order. `extra` goes on the journey."
    if not docs: raise ValueError('no pages to walk')
    got, subject = {}, clean_subject(subject)
    for k, d in enumerate(docs):
        if progress: progress(k, len(docs))
        got[d.doc_id] = from_vault(lr, ask, d.doc_id, subject, d.shelf, chars=val('links.read'))
    told = lambda s, d: bool((placement(lr.concept(s) or {}, d.doc_id) or {}).get('passage'))
    named = lambda s: [d for d in docs if s in got[d.doc_id]]
    home = {s: (first(d for d in named(s) if told(s, d)) or named(s)[0]).doc_id for d in docs for s in got[d.doc_id]}
    steps = [x for d in docs for x in _steps(lr, [s for s in got[d.doc_id] if home[s] == d.doc_id], d.doc_id, d.shelf, d.title, tool)]
    title = title or docs[0].title
    return _save_walk(lr, ask, _new_id(title), title, steps, dict(url=docs[0].url, docs=[dict(d) for d in docs]), f'Read {title}', _subject_of(lr, [s for v in got.values() for s in v], subject), tool, **extra)

`walk_from_topic` plans a path to the goal. Then each step reads from the first passage in your sources that names the idea.
A step has no `read` when no passage names it.

In [ ]:
#| export
def _topic_read(c, sources) -> dict|None:
    "The first passage in `sources` that names concept `c`, as a `read`, or None."
    name = c.get('name') or c.slug
    p = first(ground(name, sources, 1, must=[name, *(c.get('aliases') or ())])) if sources else None
    return p and dict(src=p.src, ref=p.ref, title=p.title, url=p.url, text=p.text[:val('walks.passage')], section='', figure=None)

def _topic_walk(lr:Learning, j, sources=None, tool:bool=False, **extra) -> AttrDict:
    "Save journey `j` as a walk of its first `walks.steps` steps, in its order, with a passage from `sources` and a check on each step. The other steps go in `more`, and `extra` goes on the walk."
    most = val('walks.steps')
    j.steps, more = j.steps[:most], [s['slug'] for s in j.steps[most:]]
    for s in j.steps:
        c = lr.concept(s['slug'])
        s.update(read=_topic_read(c, sources), check=_check(lr, c, tool), **_leela(c, tool))
    cap_hands(lr, j.steps, tool)
    return lr.put_journey({**j, 'mode': 'walk', 'source': None, 'started': None, 'sitting': None, 'tool': tool, 'more': more, **extra})

def walk_from_topic(lr:Learning, ask, goal:str, who:str='', sources=None, tool:bool=False) -> AttrDict:
    "Plan a path to `goal` and make it a walk of at most `walks.steps` steps, with a passage from `sources` and a check on each step."
    return _topic_walk(lr, plan(lr, ask, goal, who, sources), sources, tool)

## Change a step, read the rail

`walk_step` merges changes into one step and saves the walk. `merge_journey` merges changes into the fields of the walk, such as `sitting`. Both hold one lock for each journey id, so two requests on one walk do not lose a change. `walk_state` gives each step a dot for the rail: `done` (outcome `solid`),
`shaky` (outcome `fragile`, `gap` or `misconception`), `skipped`, `known`, `now` (the first step with no outcome and the status `todo`)
or `todo`.

In [ ]:
#| export
_JLOCKS, _JLOCK = {}, threading.Lock()

def _jlock(jid) -> threading.RLock:
    "The lock of journey `jid`, so that two writers do not read, change and write it at the same time."
    with _JLOCK: return _JLOCKS.setdefault(jid, threading.RLock())

@patch
def walk_step(self:Learning, jid:str, i:int, **kv) -> AttrDict:
    "Merge `kv` into step `i` of walk `jid` and save it, under the lock of the journey."
    with _jlock(jid):
        j = self.journey(jid)
        if j is None: raise KeyError(f'no journey {jid!r}')
        if not -len(j.steps) <= i < len(j.steps): raise IndexError(f'no step {i} in journey {jid!r}')
        j.steps[i].update(kv)
        return self.put_journey(j)

@patch
def merge_journey(self:Learning, jid:str, **kv) -> AttrDict:
    "Merge the fields `kv` into journey `jid` and save it, under the lock of the journey. The other fields stay as they are."
    with _jlock(jid):
        j = self.journey(jid)
        if j is None: raise KeyError(f'no journey {jid!r}')
        return self.put_journey({**j, **kv})

SHAKY = ('fragile', 'gap', 'misconception')

def _dot(s, now:bool) -> str:
    o = (s.get('done') or {}).get('outcome')
    if o == 'solid': return 'done'
    if o in SHAKY: return 'shaky'
    if o == 'skipped' or s.get('status') == 'skip': return 'skipped'
    if s.get('status') == 'known': return 'known'
    return 'now' if now else 'todo'

def walk_state(j) -> AttrDict:
    "The steps with a `dot`, the index of the current step, the counts, and the sections as `(name, [indexes])`."
    cur = next((i for i, s in enumerate(j['steps']) if not s.get('done') and s.get('status', 'todo') == 'todo'), None)
    steps = [AttrDict(s, dot=_dot(s, i == cur)) for i, s in enumerate(j['steps'])]
    secs, last = [], None
    for i, s in enumerate(steps):
        r = s.get('read') or {}
        name = r.get('section') or ''
        if secs and last == (r.get('doc_id'), name): secs[-1][1].append(i)
        else: secs.append((name, [i]))
        last = (r.get('doc_id'), name)
    return AttrDict(steps=steps, current=cur, done=sum(s.dot in ('done', 'known') for s in steps),
                    total=sum(s.dot != 'skipped' for s in steps), sections=secs)

`minutes` guesses the time for one step: reading at 200 words a minute, plus the time of its check.

In [ ]:
#| export
STEP_MIN = dict(short=2, order=2, demo=3, code=6)

def minutes(step) -> int:
    "The minutes for one step: its passage at 200 words a minute, plus the time of its check."
    words = len(((step.get('read') or {}).get('text') or '').split())
    return math.ceil(words / 200) + STEP_MIN.get((step.get('check') or {}).get('kind'), STEP_MIN['short'])

## Rabbit holes

A step can need an idea that you have not learned. For example, the scaling by √dk in a paper on attention needs gradients. These functions find what is missing, make a short side walk on it, and link the side walk back to the step.

`needs` lists the prereqs of a concept, then its gaps. A need is met at depth `READY` (3) or more, as in `model.ready`.

In [ ]:
#| export
READY = 3   # the depth at which a prereq is ready

def _by(lr, by): return by if by is not None else {c.slug: c for c in lr.concepts()}

def needs(lr:Learning, slug:str, by:dict=None) -> L:
    "The prereqs, then the gaps, of concept `slug` that are on the shelf. `ok` is true at depth `READY` or more."
    by = _by(lr, by); c = by[slug]
    ps = c.get('prereqs') or []
    return L(AttrDict(slug=s, name=by[s].name, floor=by[s].get('floor', 0), why=why, ok=by[s].get('floor', 0) >= READY)
             for s, why in [*((p, 'prereq') for p in ps), *((g, 'gap') for g in c.get('gaps') or () if g not in ps)] if s in by)

`pick_way_down` picks how to learn a missing idea:

- `primer`: one step and one short check. You met the idea before (depth 1 or 2), or it needs no other idea that you have not learned.
- `walk`: a side walk on the chain, at most `walks.chain` (4 by default) ideas. The chain is the unready prereqs, deepest first, then the idea.
- `session`: a longer chain. It opens as its own walk.

In [ ]:
#| export
def _chain(slug, by, seen) -> list:
    "The unready prereqs of `slug`, deepest first, then `slug`."
    seen.add(slug); out = []
    for p in by[slug].get('prereqs') or ():
        if p in by and p not in seen and by[p].get('floor', 0) < READY: out += _chain(p, by, seen)
    return out + [slug]

def _ideas(n): return f"{n} {'idea' if n == 1 else 'ideas'} that you have not learned yet"

def pick_way_down(lr:Learning, slug:str, by:dict=None) -> AttrDict:
    "How to learn concept `slug` before you go on: a `primer`, a side `walk` or its own `session`, with why in one line."
    by = _by(lr, by)
    if slug not in by: raise KeyError(f'no concept {slug!r}')
    chain = _chain(slug, by, set())
    if by[slug].get('floor', 0) in (1, 2): how, why = 'primer', 'you met it before, so a short primer is enough'
    elif len(chain) == 1: how, why = 'primer', 'it needs no other idea that you have not learned'
    elif len(chain) <= val('walks.chain'): how, why = 'walk', f'it builds on {_ideas(len(chain) - 1)}'
    else: how, why = 'session', f'it builds on {_ideas(len(chain) - 1)}, too many for a side walk'
    return AttrDict(how=how, why=why, chain=chain)

`detour` makes the side walk and starts it. It keeps `parent` (the walk and the step that opened it) and `depth`. A side walk of a side walk has depth 2. Past `walks.depth` (2 by default), only its own session can open, at depth 0. With no chain on the shelf, the side walk is a short path that `plan` makes, with steps as `walk_from_topic` makes them. The planned journey is not kept. It always ends on the idea.

In [ ]:
#| export
def _step(lr:Learning, c, sources, check, why:str='') -> dict:
    "A walk step on concept `c`, as `walk_from_topic` makes it, with the check that `check` gives."
    return dict(slug=c.slug, status='todo', why=why, read=_topic_read(c, sources), check=check(lr, c))

def _topic_steps(lr:Learning, ask, c, goal:str, most, sources) -> list:
    "The steps of a path to `goal`, at most `most` of them, then a step on `c`. The planned journey is not kept."
    t = plan(lr, ask, goal, sources=sources)
    lr.drop_journey(t.id)
    ss = [s for s in t.steps if s['slug'] != c.slug][:most]
    return [_step(lr, lr.concept(s['slug']), sources, _check, s.get('why', '')) for s in ss] + [_step(lr, c, sources, _check)]

def detour(lr:Learning, ask, slug:str, parent:dict, how:str='walk', sources=None) -> AttrDict:
    "Make and start a side walk on concept `slug`, as a child of `parent` (`{walk, step}`, or a place in Leela)."
    c, pj = lr.concept(slug), lr.journey(parent['walk']) if 'walk' in parent else None
    if c is None: raise KeyError(f'no concept {slug!r}')
    if 'walk' in parent and pj is None: raise KeyError(f"no journey {parent['walk']!r}")
    depth = 0 if how == 'session' else (pj or {}).get('depth', 0) + 1
    if depth > (most := val('walks.depth')): raise ValueError(f'too deep: a side walk at depth {depth}, and the limit is {most}. Open it as its own session.')
    of = lr.concept(pj.steps[parent['step']]['slug']).name if pj else ''
    goal = f"{c.name}: the basics" + (f", enough to read {of}" if of else '')
    if how == 'primer': steps = [_step(lr, c, sources, short_check)]
    elif len(chain := pick_way_down(lr, slug).chain) > 1:
        steps = [_step(lr, lr.concept(s), sources, _check) for s in (chain[-val('walks.chain'):] if how == 'walk' else chain)]
    elif how == 'walk': steps = _topic_steps(lr, ask, c, goal, val('walks.chain') - 1, sources)
    else: steps = _topic_steps(lr, ask, c, f'{c.name} from the start', None, sources)
    return lr.put_journey(dict(id=_new_id(c.name), title=c.name, goal=goal, subject=c.get('subject', ''), steps=steps, created=time.time(),
                               mode='walk', source=None, started=time.time(), sitting=None, parent=dict(parent), depth=depth, slug=slug))

`parent_step` gives the walk and the step that opened side walk `j`, when both are still there. Else it gives `(None, None)`. The walk page, the side pane, the history and the journey list use it, so they agree on "the parent step is still there". With `js`, a dictionary of journeys by id, it reads no journey from the shelf.

In [ ]:
#| export
def parent_step(lr:Learning, j, js:dict=None) -> tuple:
    "The parent walk and step of side walk `j`, when the walk is there and has that step; else `(None, None)`. `js` maps ids to journeys that are already read."
    p = (j or {}).get('parent') or {}
    w, i = p.get('walk'), p.get('step')
    pj = (js.get(w) if js is not None else lr.journey(w)) if isinstance(w, str) else None
    return (pj, i) if pj and isinstance(i, int) and not isinstance(i, bool) and 0 <= i < len(pj.steps) else (None, None)

`trail` gives the walks from the root walk down to a side walk, for the crumbs. `back_from` keeps the result of a side walk on the step that opened it, in `detours`. The result is the outcome of the last step, or `skipped`. One side walk has one record. When you come back again with a new outcome, the record changes, and `at` is the time of the new outcome. A later skip does not change a record.

In [ ]:
#| export
def trail(lr:Learning, jid:str) -> L:
    "The walks from the root walk to walk `jid`, each with the step that you are on."
    j, out = lr.journey(jid), L()
    step = walk_state(j).current if j else None
    while j and j.id not in out.attrgot('walk'):
        out.insert(0, AttrDict(walk=j.id, title=j.title, step=step))
        p = j.get('parent') or {}
        step, j = p.get('step'), lr.journey(p['walk']) if 'walk' in p else None
    return out

def back_from(lr:Learning, jid:str) -> AttrDict|None:
    "Keep the result of side walk `jid` on the step that opened it: one record for each side walk, updated when its outcome changes. Return the parent walk, or None."
    j = lr.journey(jid)
    pj, i = parent_step(lr, j)
    if pj is None: return None
    ds = pj.steps[i].get('detours') or []
    s = next((s for s in reversed(j.steps) if s['slug'] == j.get('slug')), {})
    d = dict(walk=jid, slug=j.get('slug'), outcome=(s.get('done') or {}).get('outcome') or 'skipped', at=time.time())
    k = next((k for k, x in enumerate(ds) if x['walk'] == jid), None)
    if k is None: return lr.walk_step(pj.id, i, detours=[*ds, d])
    if d['outcome'] in ('skipped', ds[k]['outcome']): return pj   # nothing new: never back to skipped
    return lr.walk_step(pj.id, i, detours=[*ds[:k], d, *ds[k + 1:]])

`for_talk` tells a walk that is for the room: a side walk that a talk opened (its parent is `{talk, slide}`), or the walk that a talk
was made from, from a plain journey (its parent is `{journey, for: 'talk'}`). Such a walk is not your own learning, so the
pages that list your walks and the steps you read skip it.

In [ ]:
#| export
def for_talk(j) -> bool:
    "Whether walk `j` is for a talk: a side walk that a talk opened beside a slide, or the walk that a talk was made from, from a journey."
    p = (j or {}).get('parent')
    return isinstance(p, dict) and ('talk' in p or p.get('for') == 'talk')

A walk with a talk parent, or one made for a talk from a journey, is for the room. A walk with a walk parent, or with no parent, is your own.

In [ ]:
assert for_talk(dict(parent={'talk': 't-1', 'slide': 's3'})) and not for_talk(dict(parent={'walk': 'w-1', 'step': 0}))
assert for_talk(dict(parent={'journey': 'j-1', 'for': 'talk'})) and not for_talk(dict(parent={'journey': 'j-1'}))
assert not for_talk({}) and not for_talk(None) and not for_talk(dict(parent=None))

### A journey as a talk

A presenter can give a plain journey to a group, too. `walk_for_talk` makes a walk from the journey: the same steps in the same
order, with the goal of the journey. It builds each step as `walk_from_topic` does, with a passage from your sources and a check.
A journey with more than `walks.steps` steps gives a walk of the first steps. The other steps go in `more`. The walk has the
parent `{journey, for: 'talk'}`, so `for_talk` keeps it out of your own learning. A second call gives the same walk, and
`again=True` makes a new one. `journey_walk` finds the newest walk that a journey gave.

In [ ]:
#| export
def _to_present(lr:Learning, j) -> list:
    "The steps of journey `j` that a talk presents: not skipped, and with a concept on the shelf."
    return [s for s in j.steps if s.get('status') != 'skip' and lr.concept(s['slug'])]

def journey_walk(lr:Learning, jid:str) -> AttrDict|None:
    "The newest walk that `walk_for_talk` made from journey `jid`, or None."
    ws = [j for j in lr.journeys() if j.get('parent') == {'journey': jid, 'for': 'talk'}]
    return first(sorted(ws, key=lambda j: -(j.get('created') or 0)))

def walk_for_talk(lr:Learning, jid:str, sources=None, again:bool=False) -> AttrDict:
    "A walk for a talk from plain journey `jid`: its steps that are not skipped, in order, with its goal. `made_from` keeps their slugs. A second call gives the same walk, unless `again`."
    j = lr.journey(jid)
    if j is None or j.get('mode') == 'walk' or for_talk(j): raise KeyError(f'no plain journey {jid!r}')
    if not again and (w := journey_walk(lr, jid)): return w
    ss = [dict(slug=s['slug'], status='todo', why=s.get('why', '')) for s in _to_present(lr, j)]
    if not ss: raise ValueError(f'journey {jid!r} has no step to present')
    w = AttrDict(id=_new_id(j.title), title=j.title, goal=j.get('goal', ''), subject=j.get('subject', ''), steps=ss, created=time.time())
    return _topic_walk(lr, w, sources, parent={'journey': jid, 'for': 'talk'}, made_from=[s['slug'] for s in ss])

The tests use a temporary shelf and the `Fake` model.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake

Make a plain journey with four steps. You added the last step, and you skip the third.

In [ ]:
pl = temp_learning()
for n in ('Neurons', 'Weights', 'Activation', 'Backpropagation'): pl.put(n, f'{n} is one part of a network.', 'ml')
pj = pl.put_journey(dict(id='how-a-net-works-ab12', title='How a Neural Network Works', goal='Learn how a neural network works', subject='ml',
                         steps=[dict(slug='neurons', status='todo', why='A net is made of neurons.'), dict(slug='weights', status='known', why='Each input has a weight.'),
                                dict(slug='activation', status='skip', why='')], created=1.))
pj = pl.journey_add(pj.id, 'backpropagation', why='you added it')

The walk has the steps of the journey in order, with no skipped step, and the goal of the journey. It is for a talk.

In [ ]:
pw = walk_for_talk(pl, pj.id)
test_eq([s['slug'] for s in pw.steps], ['neurons', 'weights', 'backpropagation'])
test_eq((pw.mode, pw.goal, pw.title, pw.more, pw.parent), ('walk', pj.goal, pj.title, [], {'journey': pj.id, 'for': 'talk'}))
assert for_talk(pw) and all(s['check'] and s['status'] == 'todo' for s in pw.steps) and pw.steps[0]['why'] == 'A net is made of neurons.'
assert pl.journey(pj.id).get('mode') is None and len(pl.journey(pj.id).steps) == 4   # the journey does not change

A second call gives the same walk. `again=True` makes a new walk, and `journey_walk` then gives the new one.

In [ ]:
test_eq(walk_for_talk(pl, pj.id).id, pw.id)
pw2 = walk_for_talk(pl, pj.id, again=True)
assert pw2.id != pw.id and journey_walk(pl, pj.id).id == pw2.id and journey_walk(pl, 'nope') is None

A journey with more than `walks.steps` steps gives the first steps; the others go in `more`. A walk is not a plain journey.

In [ ]:
for k in range(14): pl.put(f'Part {k}', f'Part {k} is one idea.', 'ml')
lj = pl.put_journey(dict(id='long-ab12', title='Long', goal='Learn many parts', steps=[dict(slug=f'part-{k}', status='todo', why='') for k in range(14)]))
lw = walk_for_talk(pl, lj.id)
test_eq((len(lw.steps), lw.more), (12, ['part-12', 'part-13']))
test_fail(lambda: walk_for_talk(pl, pw.id), contains='no plain journey')
test_fail(lambda: walk_for_talk(pl, 'nope'), contains='no plain journey')

`journey_changed` tells when the journey has other steps now, or the same steps in another order, than when its walk for a talk was made. Then the talk is out of date.

In [ ]:
#| export
def journey_changed(lr:Learning, w) -> bool:
    "Whether the journey that walk `w` was made from for a talk has other steps now, or the same steps in another order."
    p, was = (w or {}).get('parent') or {}, (w or {}).get('made_from')
    j = lr.journey(p.get('journey') or '') if p.get('for') == 'talk' else None
    return bool(j and was is not None and [s['slug'] for s in _to_present(lr, j)] != was)

The newest walk matches the journey. A move of a step changes the journey. A walk that is not for a talk has no journey to change.

In [ ]:
assert not journey_changed(pl, pw2)
pl.journey_step(pj.id, 'neurons', 'down')
assert journey_changed(pl, pl.journey(pw2.id)) and not journey_changed(pl, pl.journey(pj.id)) and not journey_changed(pl, None)

`look_up` finds a term that you select. A term that is not on the shelf becomes a gap of the step, through `name_gaps`.

In [ ]:
#| export
def look_up(lr:Learning, ask, term:str, at:str='', sources=None) -> AttrDict|None:
    "Find `term` on the shelf by slug, name or alias. Else add it as a gap of the concept of step `at` (`'<walk id>/<step>'`)."
    slug = _index(lr).get(term.strip().lower())
    if not slug and at:
        jid, i = at.rsplit('/', 1)
        slug = first(name_gaps(lr, ask, lr.journey(jid).steps[int(i)]['slug'], [term], sources))
    if not slug: return None
    c = lr.concept(slug)
    return AttrDict(slug=c.slug, name=c.name, statement=c.get('statement', ''))

## Add a topic, split a step

`add_topic` adds a topic by name. When the shelf knows the name, `add_topic` uses that concept. For a new name, the model writes a short statement.

In [ ]:
#| export
PRIMER_SP = register('primer', 'Statement of a new idea', 'How the tutor writes the statement of an idea that it adds to a path or looks up for you.',
                     "You write one true, self-contained statement of one idea, for a learner: one to three sentences. Maths in $...$.")

def add_topic(lr:Learning, ask, jid:str, name:str, before:str=None, sources=None) -> AttrDict:
    "Add topic `name` to journey `jid`, as a concept on the shelf. Return the journey."
    j = lr.journey(jid)
    slug = _index(lr).get(name.strip().lower())
    if not slug:
        r = ask_json(for_task(ask, 'extract'), f"The idea: {name}\nThe path it is part of: {j.title} ({j.get('goal', '')})\n\n"
                          'Return JSON: {"name": str, "statement": str}', prompt('primer'))
        slug = lr.put(r.get('name') or name, r.get('statement', ''), j.get('subject', ''), provenance=[dict(src='journey', journey=jid)]).slug
        if sources:
            try: lr.connect(slug, sources)
            except Exception: pass
    return lr.journey_add(jid, slug, before, why='you added it')

`name_gaps` keeps each idea that you say you do not know as a gap of the concept you learn. It finds the idea on the shelf by name or alias. Or it adds the idea with a short statement. Next time the gap comes first, but it does not block the concept.

In [ ]:
#| export
def name_gaps(lr:Learning, ask, slug:str, terms, sources=None) -> L:
    "Find or add a concept for each idea in `terms`, and keep it as a gap of `slug`. Return the gap slugs."
    c, idx, out = lr.concept(slug), _index(lr), L()
    for t in dict.fromkeys(str(t).strip() for t in terms or () if str(t).strip()):
        if not (g := idx.get(t.lower())):
            r = ask_json(for_task(ask, 'extract'), f"The idea: {t}\nIt comes up in: {c.name} ({c.get('subject') or 'general'})\n\n"
                              'Return JSON: {"name": str, "statement": str}', prompt('primer'))
            name = r.get('name') or t
            g = lr.put(name, r.get('statement', ''), c.get('subject', ''), aliases=[t] if t.lower() != name.lower() else [],
                       provenance=[dict(src='gap', of=slug, said=t, at=time.time())]).slug
            if sources:
                try: lr.connect(g, sources)
                except Exception: pass
        lr.add_gap(slug, g); out.append(g)
    return out

A new idea becomes a concept with a statement and the gap provenance. `name_gaps` finds an idea that is already on the shelf by its alias. It does not add that idea again.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
ngl = temp_learning(); ngl.put('Functions and the neuron', 'z = w . x + b.', 'ml'); ngl.put('Bias', 'A constant added to z.', 'ml', aliases=['the bias'])
gask = Fake({'The idea: the loop': dict(name='Weighted sum as a loop', statement='A loop adds each weight times its input.')})
test_eq(name_gaps(ngl, gask, 'functions-and-the-neuron', ['the loop', 'the bias', 'the loop']), ['weighted-sum-as-a-loop', 'bias'])
g = ngl.concept('weighted-sum-as-a-loop')
test_eq((g.aliases, g.provenance[0]['src'], g.provenance[0]['of']), (['the loop'], 'gap', 'functions-and-the-neuron'))
test_eq(ngl.concept('functions-and-the-neuron').gaps, ['weighted-sum-as-a-loop', 'bias'])

`split` divides a big step into smaller steps. It runs `extract` over the statement of the step and the passages from the
learner's sources. The concepts that it finds go before the step, as its prereqs.

In [ ]:
#| export
def split(lr:Learning, ask, jid:str, slug:str, sources=None) -> L:
    "Find the parts of step `slug`, add them before it as steps and prereqs, and return their slugs."
    c, j = lr.concept(slug), lr.journey(jid)
    ps = ground(' '.join([c.name, *(c.aliases or [])][:4]), sources, 4) if sources else L()
    text = f"{c.name}\n\n{c.statement}" + (f"\n\nPassages:\n{ground_text(ps)}" if ps else '')
    have = {s['slug'] for s in j.steps}
    new = L(s for s in extract(lr, ask, text, c.subject, dict(src='journey', journey=jid, split=slug)) if s != slug and s not in have)
    for s in new:
        lr.link(slug, s)
        lr.journey_add(jid, s, before=slug, why=f'a part of {c.name}')
    return new

## Related topics

`related` gives the concepts next to `slug` in the graph. First come the concepts that build on it, then the related ones, then its
prereqs. The concepts that are not learned come first. `with_step` finds the journeys that have a concept. `beyond` gives what comes
after a journey: the concepts that build on its steps and are not in it.

In [ ]:
#| export
def related(lr:Learning, slug:str, k:int=6, goal:int=None, by:dict=None) -> L:
    "Up to `k` concepts next to `slug`: what builds on it, what relates to it, then what it needs."
    by = by if by is not None else {c.slug: c for c in lr.concepts()}
    c = by.get(slug)
    if c is None: return L()
    up = [x for x in by.values() if slug in (x.get('prereqs') or ()) or slug in (x.get('related') or ())]
    near = [*up, *(by[s] for s in c.get('related') or () if s in by), *(by[s] for s in c.get('prereqs') or () if s in by)]
    seen, out = {slug}, []
    for x in sorted(near, key=lambda x: learned(x, goal)):   # a stable sort keeps the order of kinds
        if x.slug not in seen: seen.add(x.slug); out.append(x)
    return L(out[:k])

def with_step(lr:Learning, slug:str) -> L:
    "`(journey, index)` for each journey that has step `slug`."
    return L((j, i) for j in lr.journeys() for i, s in enumerate(j.steps) if s['slug'] == slug)

def beyond(lr:Learning, j, k:int=5, goal:int=None) -> L:
    "Up to `k` concepts that build on the steps of journey `j` and are not in it."
    by, have = {c.slug: c for c in lr.concepts()}, {s['slug'] for s in j['steps']}
    out = L()
    for s in j['steps'][::-1]:
        out += [x for x in related(lr, s['slug'], k, goal, by) if x.slug not in have and x.slug not in out.attrgot('slug')]
    return out[:k]

## Tests

A fake model plans a path to LLMs. The shelf has `attention` already, at depth 3.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Attention', 'Weights from query-key dot products.', 'ml')
lr.update('attention', floor=3, seen=1)
fake = Fake({'You plan a learning path': dict(title='How LLMs work', subject='ml', steps=[
    dict(name='Tokenizers', statement='A tokenizer cuts text into tokens.', slug='', why='The model reads tokens.', needs=[]),
    dict(name='Embeddings', statement='An embedding maps a token to a vector.', slug='', why='Tokens become vectors.', needs=['Tokenizers']),
    dict(name='Attention', statement='', slug='attention', why='The core of a transformer.', needs=['Embeddings']),
    dict(name='Mamba', statement='Mamba is a selective state-space model.', slug='', why='Another way to mix tokens.', needs=['Attention'])])})
j = plan(lr, fake, 'I want to learn how LLMs work', who='Background: codes in Python')

`plan` keeps the order, uses the known concept, and makes the new ones with their prereqs.

In [ ]:
test_eq([s['slug'] for s in j.steps], ['tokenizers', 'embeddings', 'attention', 'mamba'])
test_eq((lr.concept('embeddings').prereqs, lr.concept('mamba').prereqs, lr.concept('mamba').subject), (['tokenizers'], ['attention'], 'ml'))
test_eq(lr.concept('attention').floor, 3)    # a known concept keeps its state
assert 'codes in Python' in fake.calls[-1][0] and 'attention: Attention (depth 3)' in fake.calls[-1][0]

`progress` names the state of each step. The current step is the first that is not done.

In [ ]:
pg = progress(lr, j)
test_eq([s.state for s in pg.steps], ['new', 'new', 'started', 'new'])
test_eq((pg.done, pg.total, pg.current.slug), (0, 4, 'tokenizers'))

A step that the learner learns anywhere is learned in this journey too. A step that is marked known or skipped is done. Then `current` moves to the next step.

In [ ]:
lr.update('tokenizers', floor=4, seen=2)
lr.journey_step(j.id, 'embeddings', 'known')
pg = progress(lr, lr.journey(j.id))
test_eq([s.state for s in pg.steps], ['learned', 'known', 'started', 'new'])
test_eq((pg.done, current(lr, lr.journey(j.id)), current(lr, lr.journey(j.id), skip={'attention'})), (2, 'attention', 'mamba'))

Steps move up and down. A removed step leaves the path. A skipped step does not count in the total.

In [ ]:
lr.journey_step(j.id, 'mamba', 'up')
test_eq([s['slug'] for s in lr.journey(j.id).steps], ['tokenizers', 'embeddings', 'mamba', 'attention'])
lr.journey_step(j.id, 'mamba', 'skip')
test_eq(progress(lr, lr.journey(j.id)).total, 3)
lr.journey_step(j.id, 'mamba', 'remove')
test_eq(len(lr.journey(j.id).steps), 3)

`add_topic` uses a known name, and asks the model only for a new one.

In [ ]:
fake.rules['The idea: '] = dict(name='PyTorch', statement='PyTorch is a Python library for tensors and autograd.')
add_topic(lr, fake, j.id, 'PyTorch')
add_topic(lr, fake, j.id, 'tokenizers')    # known, and in the path already: no change
test_eq([s['slug'] for s in lr.journey(j.id).steps], ['tokenizers', 'embeddings', 'attention', 'pytorch'])

`split` adds the parts of a step before it, as its prereqs.

In [ ]:
fake.rules['Return JSON: {"concepts"'] = dict(concepts=[dict(name='Query and key', statement='Each token has a query and a key.', aliases=[], prereqs=[])])
new = split(lr, fake, j.id, 'attention')
test_eq(list(new), ['query-and-key'])
test_eq([s['slug'] for s in lr.journey(j.id).steps], ['tokenizers', 'embeddings', 'query-and-key', 'attention', 'pytorch'])
assert 'query-and-key' in lr.concept('attention').prereqs

`related` puts what builds on a concept first. `with_step` finds the journeys of a concept. `beyond` gives what comes after.

In [ ]:
lr.put('Transformer block', 'Attention, then a feed-forward layer.', 'ml', prereqs=['attention'])
test_eq(related(lr, 'attention').attrgot('slug'), ['mamba', 'transformer-block', 'query-and-key'])   # mamba left the path, but it builds on attention
test_eq([(x.id, i) for x, i in with_step(lr, 'attention')], [(j.id, 3)])
test_eq(beyond(lr, lr.journey(j.id)).attrgot('slug'), ['mamba', 'transformer-block'])

`journeys` lists the journeys with the newest change first. `drop_journey` removes one journey.

In [ ]:
j2 = lr.put_journey(dict(id='sanskrit', title='Read the Gita', goal='read', steps=[]))
test_eq(lr.journeys().attrgot('id'), ['sanskrit', j.id])
lr.drop_journey('sanskrit')
test_eq(len(lr.journeys()), 1)

## Tests for walks

A walk is a journey with `mode='walk'`. These tests use the small paper in `tests/walk_data.py` and a fake model.

In [ ]:
#| hide
import sys, abhyasa
from pathlib import Path
sys.path.insert(0, str(Path(abhyasa.__file__).parents[1]/'tests'))
from walk_data import *
from abhyasa.ground import passage
from abhyasa.walk import tutor_note, step_meta
from abhyasa.prompts import PROMPTS, set_prompt

A walk from a source has one step for each idea of the paper, in the order of the paper. Each step has the passage to read and a check.

In [ ]:
wl = temp_learning(); did = add_paper(wl)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY})
wj = journey_from_source(wl, ask, did)
assert wj.mode == 'walk' and wj.source == dict(shelf='store', doc_id=did, title='Attention Is All You Need') and wj.started is None
assert [s['slug'] for s in wj.steps] == ['sequential-computation', 'softmax', 'scaling-by-the-root-of-dk']
r = wj.steps[1]['read']
assert (r['section'], r['page'], r['figure']) == ('3.2.1 Scaled Dot-Product Attention', 4, 'attention/images/fig2.png')
assert wj.steps[1]['check']['kind'] in ('code', 'short') and wj.steps[2]['check']['way'] in ('demo', 'maths')
assert progress(wl, wj).total == 3     # progress does not change

`walk_state` gives each step a dot for the rail. `walk_step` keeps a change on one step.

In [ ]:
ws = walk_state(wl.walk_step(wj.id, 0, done=dict(outcome='fragile', aid='a1', at=1.)))
assert [s.dot for s in ws.steps] == ['shaky', 'now', 'todo'] and ws.current == 1
assert ws.sections == [('1 Introduction', [0]), ('3.2.1 Scaled Dot-Product Attention', [1, 2])]
assert minutes(wj.steps[1]) >= 2
test_fail(lambda: wl.walk_step('none', 0), contains='none', exc=KeyError)
test_fail(lambda: wl.walk_step(wj.id, 9), exc=IndexError)

`merge_journey` changes only the fields that it names. The steps stay as they are, and an unknown walk raises `KeyError`.

In [ ]:
before = wl.journey(wj.id).steps
test_eq(wl.merge_journey(wj.id, sitting='s1').sitting, 's1')
test_eq(wl.journey(wj.id).steps, before)
test_fail(lambda: wl.merge_journey('none', sitting='s1'), contains='none', exc=KeyError)

A concept that you know at depth 4 or more stays in the walk, as known.

In [ ]:
wl.update('softmax', floor=4)
assert journey_from_source(wl, ask, did).steps[1]['status'] == 'known'

A walk from pasted text takes the passage only when its quote is in the text. An idea with no quote in the text has no passage.

In [ ]:
jt = journey_from_text(wl, ask, PAGES[1][1], 'Attention notes')
sm = next(s for s in jt.steps if s['slug'] == 'softmax')
assert jt.source is None and sm['read']['text'].startswith('apply a softmax function')
assert next(s for s in jt.steps if s['slug'] == 'sequential-computation')['read']['text'] == ''

A walk from a topic plans the path, then reads from the first passage in your sources that names each idea. With no sources, a step has no passage.

In [ ]:
PLAN = dict(title='Attention', subject='ml', steps=[dict(name='Softmax', statement='', slug='softmax', why='Weights.', needs=[], ways=['code'])])
class Notes:
    name = 'notes'
    def search(self, q, k=4): return [passage('vault:notes', 'n1', 'My notes', 'Softmax turns scores into weights.')]
jw = walk_from_topic(wl, Fake({'You plan a learning path': PLAN}), 'learn attention', sources=[Notes()])
assert jw.mode == 'walk' and jw.steps[0]['read']['text'].startswith('Softmax turns') and jw.steps[0]['check']
assert walk_from_topic(wl, Fake({'You plan a learning path': PLAN}), 'learn attention', sources=[]).steps[0]['read'] is None

A walk across two pages has one step for each idea, in page order. Softmax is named in both pages. Its step is in the page where its passage is.

In [ ]:
lr = temp_learning()
a = lr.vault.add(PAGES[0][1], 'Introduction', source='https://example.dev/docs/intro', kind='web')
b = lr.vault.add([(0, PAGES[1][1]), (1, PAGES[2][1])], 'Attention', source='https://example.dev/docs/attention', kind='web')
docs = [AttrDict(shelf='store', doc_id=d['doc_id'], title=t, url=u) for d, t, u in
        [(a, 'Introduction', 'https://example.dev/docs/intro'), (b, 'Attention', 'https://example.dev/docs/attention')]]
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY})
j = journey_from_sources(lr, ask, docs, 'Attention docs')
test_eq([s['slug'] for s in j.steps], ['sequential-computation', 'softmax', 'scaling-by-the-root-of-dk'])
test_eq([s['read']['doc_id'] for s in j.steps], [a['doc_id'], b['doc_id'], b['doc_id']])   # softmax is named in both; its step is where its passage is
assert j.steps[1]['read']['text'].startswith('apply a softmax function') and j.source['url'] == 'https://example.dev/docs/intro'
test_eq(journey_from_sources(lr, ask, docs, 'Tool docs', tool='cli').tool, 'cli')         # extra fields go on the journey

`progress` hears of each page before the tutor reads it. Two pages with no section name stay two sections on the rail.

In [ ]:
heard = []
journey_from_sources(lr, ask, docs, progress=lambda k, n: heard.append((k, n)))
test_eq(heard, [(0, 2), (1, 2)])
test_eq(walk_state(dict(steps=[dict(read=dict(doc_id='a', section='')), dict(read=dict(doc_id='b', section=''))])).sections, [('', [0]), ('', [1])])

### The title of a paper

A paper that the PDF reader filed by the end of its link has its arXiv id as its title. `file_link` now gives the real title for a new walk. `paper_id` gives the id of the paper of a walk, for a quiet tag, when the title is not the id.

`repair_titles` fixes the old walks one time. A walk whose title is only an arXiv id takes the title of its vault document, when that has a real one. A mark on the shelf says that it ran, as in `repair_names`. It makes no network call.


In [ ]:
#| export
def _arxiv_of(j) -> str:
    "The arXiv id in the address of walk `j`, or an empty string."
    m = ARXIV_LINK.fullmatch((j.get('source') or {}).get('url') or j.get('url') or '')
    return m[1] if m else ''

def paper_id(j) -> str:
    "The arXiv id of the paper that walk `j` reads, or an empty string. Empty too when the title of `j` is that id."
    return '' if is_id_title(j.get('title')) else _arxiv_of(j)

@patch
def _retitle(self:Learning, j, t:str):
    "Give walk `j` the title `t`, and its first vault document too, and keep the place of the walk in the list."
    n = self.put_journey({**j, 'title': t, 'docs': [{**j.docs[0], 'title': t}, *j.docs[1:]]})
    self.v.set_meta(n.doc_id, updated=j.get('updated', 0))   # a repair is not a change that you made: the order of the list stays

@patch
def repair_titles(self:Learning) -> list:
    "One time for each shelf, give each walk whose title is only an arXiv id the title of its vault document, when that has a real one. Return the ids that changed."
    if first(self.v.t.docs(where='source=?', where_args=[_src('marker', 'titles')])): return []
    out = []
    for j in self.journeys():
        if not (is_id_title(j.title) and j.get('docs')): continue
        d = self.vault.shelf(j.docs[0]['shelf']).doc(j.docs[0]['doc_id'])
        if d and (t := (d['title'] or '').strip()) and not is_id_title(t):
            self._retitle(j, t); out.append(j.id)
    self.v.add('The titles of the walks were repaired.', title='Titles repaired', source=_src('marker', 'titles'), kind='marker')
    return out

A walk with an arXiv id for a title takes the title of its document. A walk with a title of its own stays, and so does a walk whose document has only the id. The repair runs one time.


In [ ]:
lr = temp_learning()
real = lr.vault.add(PAGES[0][1], 'Attention Is All You Need', source='https://arxiv.org/pdf/1706.03762', kind='pdf')
bare = lr.vault.add(PAGES[1][1], '1706.03763', source='https://arxiv.org/pdf/1706.03763', kind='pdf')
def paper_walk(jid, title, d, num):
    u = f'https://arxiv.org/pdf/{num}'
    return lr.put_journey(dict(id=jid, title=title, mode='walk', steps=[], url=u, docs=[dict(shelf='store', doc_id=d['doc_id'], title=title, url=u)]))
paper_walk('old', '1706.03762', real, '1706.03762'); paper_walk('mine', 'My title', real, '1706.03762'); paper_walk('bare', '1706.03763', bare, '1706.03763')
before = lr.journey('old').updated
lr.v.forget(first(lr.v.t.docs(where='source=?', where_args=['learning://marker/titles']))['id'])   # a vault from before the repair
test_eq(Learning(lr.vault).repair_titles(), [])   # the load did it already
test_eq([lr.journey(i).title for i in ('old', 'mine', 'bare')], ['Attention Is All You Need', 'My title', '1706.03763'])
test_eq(lr.journey('old').docs[0]['title'], 'Attention Is All You Need')
test_eq(lr.journey('old').updated, before)

In [ ]:
test_eq([paper_id(lr.journey(i)) for i in ('old', 'bare')], ['1706.03762', ''])   # no tag when the title is the id
test_eq(paper_id(dict(title='A page', url='https://docs.example.dev/')), '')
lr.put_journey({**lr.journey('old'), 'title': '1706.03762'}); test_eq(lr.repair_titles(), []); test_eq(lr.journey('old').title, '1706.03762')   # the mark is set

The first repair has no network. For a walk or a document that has only the id, `repair_paper_titles` asks arXiv for the title, with the same call as `file_link` (`arxiv_title`, which keeps the answer on disk), and a short time limit. The app runs it one time in the background after it starts, never in a load or a request.

The mark is set only when each id has a title or has failed for good (arXiv has no such paper). A failure of the network leaves the mark off, so the next start tries again.


In [ ]:
#| export
@patch
def repair_paper_titles(self:Learning, title_of=None) -> list:
    "Give each walk and each PDF in the vault that is titled with only an arXiv id the title of its paper, from `title_of(id)` (default: arXiv). Return the ids of the walks that changed. The mark is set only when no id failed from the network."
    mark = _src('marker', 'paper-titles')
    if first(self.v.t.docs(where='source=?', where_args=[mark])): return []
    title_of = title_of or partial(arxiv_title, strict=True)
    got, retry, out = {}, False, []
    def title(aid):
        nonlocal retry
        if aid not in got:
            try: got[aid] = title_of(aid)
            except Exception as e: got[aid] = None; retry = retry or 'No paper found' not in str(e)   # a paper that arXiv does not have: no more tries
        return got[aid]
    for j in self.journeys():
        if is_id_title(j.title) and j.get('docs') and (aid := _arxiv_of(j)) and (t := title(aid)): self._retitle(j, t); out.append(j.id)
    for d in self.vault.t.docs(where="kind in ('pdf', 'arxiv')"):
        if is_id_title(d['title']) and (m := ARXIV_LINK.fullmatch(d['source'] or '') or ARXIV_ID.fullmatch(d['title'])) and (t := title(ARXIV_ID.search(m[0]).group())):
            self.vault.t.docs.update(dict(id=d['id'], title=t))
    if not retry: self.v.add('The titles of the papers were repaired.', title='Paper titles repaired', source=mark, kind='marker')
    return out

The title comes from `title_of`, here a fake. A failure of the network keeps the id and leaves the mark off, so the next call tries again. A paper that arXiv does not have is not asked for again.


In [ ]:
lr = temp_learning()
d = lr.vault.add(PAGES[0][1], '1706.03762', source='https://arxiv.org/pdf/1706.03762', kind='pdf')
paper_walk('w1', '1706.03762', d, '1706.03762'); paper_walk('w2', 'My title', d, '1706.03762')
asked = []
def down(aid): asked.append(aid); raise OSError('no network')
test_eq(lr.repair_paper_titles(down), [])
test_eq((lr.journey('w1').title, first(lr.vault.t.docs(where='id=?', where_args=[d['doc_id']]))['title']), ('1706.03762', '1706.03762'))   # the id stays
test_eq(lr.repair_paper_titles(lambda aid: 'Attention Is All You Need'), ['w1'])                                                        # it tries again
test_eq((lr.journey('w1').title, lr.journey('w2').title), ('Attention Is All You Need', 'My title'))
test_eq(first(lr.vault.t.docs(where='id=?', where_args=[d['doc_id']]))['title'], 'Attention Is All You Need')
test_eq(lr.repair_paper_titles(down), [])      # the mark is set: no new call
test_eq(asked, ['1706.03762'])

In [ ]:
lr = temp_learning()
d = lr.vault.add(PAGES[0][1], '1706.03762', source='https://arxiv.org/pdf/1706.03762', kind='pdf'); paper_walk('w1', '1706.03762', d, '1706.03762')
def none(aid): raise Exception('No paper found')
test_eq(lr.repair_paper_titles(none), []); test_eq(lr.repair_paper_titles(lambda aid: 'X'), [])   # failed for good: the mark is set
test_eq(lr.journey('w1').title, '1706.03762')

A walk has at most `val('walks.steps')` steps. A source with 30 ideas gives a walk of 12 steps. The model gives 14 numbers, so only its first 12 count.
The steps keep the order of the source, but idea 9 needs idea 20, so idea 20 comes first. The other ideas stay on the shelf, and the walk keeps their slugs in `more`.

In [ ]:
LONG = ' '.join(f'Sentence number {k} of the long text is here.' for k in range(30))
MANY = dict(concepts=[dict(name=f'Idea {k}', statement=f'Idea {k} is one idea.', prereqs=['Idea 20'] if k == 9 else [],
                           quote=f'Sentence number {k} of the long text', ways=['words']) for k in range(30)])
PICK = [21, 1, 4, 10, 6, 13, 16, 19, 23, 25, 28, 30, 2, 3]   # numbers from 1
ml = temp_learning()
ask = Fake({'Return JSON: {"concepts"': MANY, 'Return JSON: {"pick"': dict(pick=PICK), 'You write a short note': 'A note.'})
mj = journey_from_text(ml, ask, LONG, 'Long notes')
test_eq(val('walks.steps'), 12)
test_eq([s['slug'] for s in mj.steps], [f'idea-{k}' for k in (0, 3, 5, 20, 9, 12, 15, 18, 22, 24, 27, 29)])
assert len(mj.more) == 18 and all(ml.concept(s) for s in mj.more)

The build makes two model calls: one finds the ideas and one picks them. Then a note is written for each of the 12 steps only.

In [ ]:
test_eq(len(ask.calls), 2)
for i in range(len(mj.steps)): tutor_note(ml, ask, mj, i)
notes = [p for p, _ in ask.calls[2:]]
test_eq(len(notes), 12)
assert not any(f'Idea: Idea {k}\n' in p for p in notes for k in (1, 2, 4))

A source with 5 ideas keeps all 5, and the model is not asked to pick. A reply that is not JSON gives the first 12 ideas. A topic plan with 14 steps keeps its first 12.

In [ ]:
few = Fake({'Return JSON: {"concepts"': dict(concepts=MANY['concepts'][:5])})
fj = journey_from_text(temp_learning(), few, LONG, 'Short notes')
test_eq(([s['slug'] for s in fj.steps], fj.more, len(few.calls)), ([f'idea-{k}' for k in range(5)], [], 1))
bad = journey_from_text(temp_learning(), Fake({'Return JSON: {"concepts"': MANY, 'Return JSON: {"pick"': 'not JSON'}), LONG, 'Long notes')
test_eq([s['slug'] for s in bad.steps], [f'idea-{k}' for k in range(12)])
BIG = dict(title='Many', subject='ml', steps=[dict(name=f'Topic {k}', statement=f'Topic {k} is one idea.', slug='', why='', needs=[], ways=['words']) for k in range(14)])
tj = walk_from_topic(temp_learning(), Fake({'Return JSON: {"title"': BIG}), 'many topics')
test_eq((len(tj.steps), tj.more), (12, ['topic-12', 'topic-13']))

`true` in the reply is not a number, so it is not idea 1. A reply with only 2 numbers keeps those 2 ideas, and the first unpicked ideas in source order fill the walk to 12 steps.

In [ ]:
pk = lambda pick: [s['slug'] for s in journey_from_text(temp_learning(), Fake({'Return JSON: {"concepts"': MANY, 'Return JSON: {"pick"': dict(pick=pick)}), LONG, 'Long notes').steps]
test_eq(pk([True, *range(2, 14)]), [f'idea-{k}' for k in range(1, 13)])
test_eq(pk([29, 30]), [f'idea-{k}' for k in (*range(10), 28, 29)])

The path planner and the pick are registered prompts, `walk.plan` and `walk.pick`. The code reads the text at each call, so a text that you save on the Settings page is used on the next walk.

In [ ]:
assert {'walk.plan', 'walk.pick'} <= set(PROMPTS)
set_prompt('walk.pick', 'Pick the main ideas. My text.'); set_prompt('walk.plan', 'Plan the path. My text.')
ask = Fake({'Return JSON: {"concepts"': MANY, 'Return JSON: {"pick"': dict(pick=PICK), 'Return JSON: {"title"': BIG})
journey_from_text(temp_learning(), ask, LONG, 'Long notes'); walk_from_topic(temp_learning(), ask, 'many topics')
assert {'Pick the main ideas. My text.', 'Plan the path. My text.'} <= {sp for _, sp in ask.calls}
set_prompt('walk.pick'); set_prompt('walk.plan')

At most a third of the steps of a walk get a hands-on check (`walks.hands`). A walk of 12 ideas that all list `code` gets 4 code checks, on its first 4 steps. The other 8 steps get a short answer in their text way.

In [ ]:
test_eq([hands_most(n) for n in (1, 2, 3, 5, 6, 12)], [0, 0, 1, 1, 2, 4])
CODE = dict(concepts=[{**c, 'ways': ['code', 'words']} for c in MANY['concepts'][:12]])
hj = journey_from_text(temp_learning(), Fake({'Return JSON: {"concepts"': CODE}), LONG, 'Code notes')
test_eq([s['check']['kind'] for s in hj.steps], ['code'] * 4 + ['short'] * 8)
assert all(s['check']['way'] == 'words' for s in hj.steps[4:])

A walk from a topic has the same cap. Here the plan has 12 topics with a code way. The first 4 get a code check, and the others get a check in words.

In [ ]:
CODE_PLAN = dict(title='Code', subject='ml', steps=[dict(name=f'Code topic {k}', statement=f'Code topic {k} is one idea.', slug='', why='', needs=[], ways=['code', 'words']) for k in range(12)])
cj = walk_from_topic(temp_learning(), Fake({'Return JSON: {"title"': CODE_PLAN}), 'code topics')
test_eq([s['check']['kind'] for s in cj.steps], ['code'] * 4 + ['short'] * 8)
assert all(s['check']['way'] == 'words' for s in cj.steps[4:])

## Tests for rabbit holes

The paper has a step on the scaling by √dk. It needs softmax, which you know, and gradients, which you do not know. `needs` lists both. `pick_way_down` picks a primer for gradients, until gradients needs an idea of its own.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
j = journey_from_source(lr, Fake({'Return JSON: {"concepts"': EXTRACT_REPLY}), did)
lr.put('Derivatives and gradients', 'The slope of a function at a point, one for each input.', 'ml', slug='gradients')
lr.update('scaling-by-the-root-of-dk', prereqs=['softmax', 'gradients']); lr.update('softmax', floor=3)
test_eq([(n.slug, n.ok) for n in needs(lr, 'scaling-by-the-root-of-dk')], [('softmax', True), ('gradients', False)])
test_eq(pick_way_down(lr, 'gradients').how, 'primer')          # no prereqs of its own: a primer is enough
lr.put('Slope of a line', '', 'ml', slug='slope'); lr.update('gradients', prereqs=['slope'])
w = pick_way_down(lr, 'gradients')
test_eq((w.how, w.chain), ('walk', ['slope', 'gradients']))     # a short chain: a side walk, deepest first

A side walk is a child of the step that opened it. Each attempt in it links back to that step. Past depth 2, only its own session can open.

In [ ]:
d = detour(lr, Fake({}), 'gradients', dict(walk=j.id, step=2))
test_eq((d.mode, d.parent, d.depth, [s['slug'] for s in d.steps]), ('walk', dict(walk=j.id, step=2), 1, ['slope', 'gradients']))
test_eq(step_meta(d, 0)['from'], f'{j.id}/2')                  # each attempt links to the step that opened it
assert 'from' not in step_meta(j, 0)                            # a walk with no parent has no `from`
test_eq([t.walk for t in trail(lr, d.id)], [j.id, d.id])
d2 = detour(lr, Fake({}), 'slope', dict(walk=d.id, step=0), how='primer')
test_eq((d2.depth, len(d2.steps), d2.steps[0]['check']['depth']), (2, 1, 2))
test_fail(lambda: detour(lr, Fake({}), 'softmax', dict(walk=d2.id, step=0), how='primer'), contains='deep')
test_eq(detour(lr, Fake({}), 'softmax', dict(walk=d2.id, step=0), how='session').depth, 0)   # past the limit: its own session

When you come back, the parent step keeps one line on the side walk. `look_up` finds a term by slug or name. A new term becomes a gap of the step.

In [ ]:
lr.walk_step(d.id, 1, done=dict(outcome='solid', aid='a1', at=1.))
p = back_from(lr, d.id); back_from(lr, d.id)
test_eq(p.id, j.id)
test_eq([(x['walk'], x['slug'], x['outcome']) for x in lr.journey(j.id).steps[2]['detours']], [(d.id, 'gradients', 'solid')])   # once
test_eq(look_up(lr, Fake({}), 'gradients').slug, 'gradients')
test_eq(look_up(lr, Fake({}), 'Derivatives and gradients').slug, 'gradients')                  # by name too
g = look_up(lr, Fake({'The idea: vanishing gradients': dict(name='Vanishing gradients', statement='Gradients near 0 stop the weights.')}),
            'vanishing gradients', at=f'{j.id}/2')
assert g.name == 'Vanishing gradients' and g.slug in lr.concept('scaling-by-the-root-of-dk').gaps   # a new term is filed as a gap of the step

A parent step that is gone gives `(None, None)`: a step past the end, a step that is not a number, a walk that is gone, or no parent.

In [ ]:
test_eq(parent_step(lr, dict(parent=dict(walk=j.id, step=2))), (lr.journey(j.id), 2))
for p in (dict(walk=j.id, step=99), dict(walk=j.id, step='2'), dict(walk='gone', step=0), None):
    test_eq(parent_step(lr, dict(parent=p)), (None, None))
test_eq(parent_step(lr, dict(parent=dict(walk=j.id, step=2)), js={}), (None, None))   # with `js`, only those journeys count

With no chain on the shelf, the side walk is a short topic walk that ends on the idea. A side walk that you leave before its last check comes back as skipped.

In [ ]:
TOPIC = dict(title='Softmax basics', subject='ml', steps=[dict(name=n, statement='', slug='', why='', needs=[]) for n in 'ABCDEF'])
tk = Fake({'You plan a learning path': TOPIC}); n = len(lr.journeys())
dt = detour(lr, tk, 'softmax', dict(walk=j.id, step=2))
test_eq([s['slug'] for s in dt.steps], ['a', 'b', 'c', 'softmax'])     # cut to 4 steps, and it ends on the idea
assert 'Softmax: the basics, enough to read Scaling by the root of dk' in tk.calls[0][0] and len(lr.journeys()) == n + 1   # no plain journey stays
test_eq(back_from(lr, dt.id).steps[2]['detours'][-1]['outcome'], 'skipped')
test_eq((back_from(lr, j.id), trail(lr, j.id).attrgot('walk')), (None, [j.id]))   # a walk with no parent
test_eq(look_up(lr, Fake({}), 'eigenvalues'), None)                                   # a new term with no step: nothing to file

Leave a side walk early, then finish it and come back again: the one record now has the outcome of the finish.

In [ ]:
at0 = lr.journey(j.id).steps[2]['detours'][-1]['at']
lr.walk_step(dt.id, 3, done=dict(outcome='fragile', aid='a2', at=2.))
ds = back_from(lr, dt.id).steps[2]['detours']
test_eq([x['outcome'] for x in ds if x['walk'] == dt.id], ['fragile'])          # one record, now with the outcome of the finish
assert ds[-1]['at'] >= at0
lr.walk_step(dt.id, 3, done=dict(outcome='skipped', at=3.))
test_eq(back_from(lr, dt.id).steps[2]['detours'][-1]['outcome'], 'fragile')       # a later skip does not undo what you learned

A parent from Leela is a notebook cell, not a walk. Its side walk has depth 1, and its attempts link to the cell.

In [ ]:
lw = detour(lr, Fake({}), 'gradients', dict(notebook='nb.ipynb', cell='c1'))
test_eq((lw.depth, step_meta(lw, 0)['from']), (1, 'nb.ipynb/c1'))
assert back_from(lr, lw.id) is None

## Open in Leela

Code with a library, tools and a code lookup are learned in Leela. abhyasa keeps only the small stdlib code check. So a walk can be marked as a tool
walk (`tool=True`), and abhyasa shows a link that opens the walk, a step or a topic in Leela. abhyasa never calls Leela. It makes a path and a query.
The learner's browser opens the link on the Leela that runs on the same computer.

`handoff` makes the path and the query. `read_handoff` is the other side, for Leela. `HANDOFF_CASES` are the cases that both test suites run.

In [ ]:
#| export
from urllib.parse import urlencode, parse_qs

LEELA_ROUTE = '/learn'
WALK_ID = re.compile(r'^[a-z0-9-]{1,48}$')   # the form of `_new_id`
TOPIC_MAX = 200

def handoff(jid:str='', i:int|None=None, topic:str='') -> str:
    "The path and query that Leela opens: a walk, a step of it, a topic, or a mix."
    if not (jid or topic): raise ValueError('nothing to open')
    q = [(k, v) for k, v in (('walk', jid), ('step', i), ('topic', topic)) if v not in ('', None)]
    return f'{LEELA_ROUTE}?{urlencode(q)}'

def read_handoff(query:str) -> AttrDict:
    "The `walk`, `step` and `topic` in `query`, checked. A `ValueError` names the key that is not valid. Other keys are ignored."
    q = {k: v[0] for k, v in parse_qs(query).items()}
    walk, step, topic = q.get('walk', ''), q.get('step'), q.get('topic', '')
    if walk and not WALK_ID.match(walk): raise ValueError('walk is not a walk id')
    if step is not None and not (step.isascii() and step.isdigit()): raise ValueError('step must be a whole number from 0')
    if len(topic) > TOPIC_MAX: raise ValueError(f'topic is more than {TOPIC_MAX} characters')
    if not (walk or topic): raise ValueError('nothing to open')
    return AttrDict(walk=walk, step=None if step is None else int(step), topic=topic)

These cases are shared. Leela runs the same list in its contract test.

In [ ]:
#| export
HANDOFF_CASES = [(dict(jid='attention-ab12'), '/learn?walk=attention-ab12'),
                 (dict(jid='attention-ab12', i=3), '/learn?walk=attention-ab12&step=3'),
                 (dict(topic='Agents and tools'), '/learn?topic=Agents+and+tools'),
                 (dict(jid='pydantic-ai-docs-9f0c', i=0, topic='Agents'), '/learn?walk=pydantic-ai-docs-9f0c&step=0&topic=Agents')]

Each case makes its path, and the path reads back to the same values. A bad id, a bad step and an empty call fail with the name of the key.

In [ ]:
for kw, path in HANDOFF_CASES:
    test_eq(handoff(**kw), path)
    test_eq(read_handoff(path.split('?', 1)[1]), AttrDict(walk=kw.get('jid', ''), step=kw.get('i'), topic=kw.get('topic', '')))
test_fail(lambda: read_handoff('walk=../x'), contains='walk')
test_fail(lambda: read_handoff('walk=a-1&step=-1'), contains='step')
test_fail(lambda: read_handoff('topic=' + 'x' * 201), contains='topic')
test_fail(lambda: read_handoff('other=1'), contains='nothing to open')
test_fail(lambda: handoff(), contains='nothing to open')

A tool walk has no code check. The step of an idea with the code way has `leela`, so the step page can point to Leela.
A walk to read does not change.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY})
j = journey_from_source(lr, ask, did, tool=True)
assert lr.journey(j.id).tool is True and j.id in [x.id for x in lr.journeys()]   # the marker does not change how a journey is kept
assert all(s['check']['kind'] != 'code' for s in j.steps)
test_eq([bool(s.get('leela')) for s in j.steps], [False, True, False])              # softmax has the code way
plain = journey_from_source(lr, ask, did)
assert not plain.tool and 'leela' not in plain.steps[1]                              # a walk to read does not change

The other makers take the same marker. A tool walk from text and from sources has no code check either.

In [ ]:
jt = journey_from_text(lr, ask, PAGES[1][1], 'Attention notes', tool=True)
assert jt.tool and all(s['check']['kind'] != 'code' for s in jt.steps) and [s['slug'] for s in jt.steps if s.get('leela')] == ['softmax']
test_eq(_check(lr, AttrDict(slug='x', name='X', ways=['code'])).way in TOOL_WAYS, False)
test_eq(_check(lr, AttrDict(slug='x', name='X', ways=['code']), tool=True).way in TOOL_WAYS, True)

A process idea gets an order check, even when a text way comes first in its list. The extract prompt tells the model when to name `order`.
A walk of 3 steps has 1 hands-on check, so the code check of softmax and the demo of the scaling become short answers. A tool walk has the same kinds.

In [ ]:
PROC = dict(concepts=[{**EXTRACT_REPLY['concepts'][0], 'ways': ['words', 'order']}, *EXTRACT_REPLY['concepts'][1:]])
ask = Fake({'Return JSON: {"concepts"': PROC})
test_eq([s['check']['kind'] for s in journey_from_source(lr, ask, did).steps], ['order', 'short', 'short'])
assert 'a process or a sequence of steps' in ask.calls[0][1]
test_eq([s['check']['kind'] for s in journey_from_source(lr, ask, did, tool=True).steps], ['order', 'short', 'short'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()